# Phase C2 Kaggle Runner
This notebook is entirely standalone. It records telemetry, clones the exact branch, runs tests and bounded benchmarks, and orchestrates the Phase C2 empty multi-scale training via the runner script.


In [ ]:
# CONFIGURATION
MODEL_TO_RUN = "M1"  # "M1" (Global-Local) or "M2" (Scalar-Only)
MAX_INTERACTIONS = 150000
RESUME = False


In [ ]:
import os
import sys
import subprocess
import psutil

# Record Telemetry
print("=== HARDWARE TELEMETRY ===")
print(f"CPU: {psutil.cpu_count(logical=True)} logical cores")
!lscpu | grep 'Model name' || echo "lscpu not available"
print("\nGPU:")
!nvidia-smi -L || echo "No GPU detected"

print("\n=== SOFTWARE TELEMETRY ===")
!python --version
print("\n=== SYSTEM RESOURCES ===")
mem = psutil.virtual_memory()
print(f"RAM: {mem.total / (1024**3):.2f} GB")
!df -h /kaggle/working

# Clone Repo Securely
print("\n=== CLONING REPOSITORY ===")
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    git_token = user_secrets.get_secret("GITHUB_TOKEN")
    repo_url = f"https://{git_token}@github.com/muzzammilsajid1/uav-dynamic-routing.git"
except:
    print("No GITHUB_TOKEN secret found. Attempting public clone...")
    repo_url = "https://github.com/muzzammilsajid1/uav-dynamic-routing.git"

!git clone --branch rl-v3-c2-empty-multiscale $repo_url repo
%cd repo
!git rev-parse HEAD

print("\n=== INSTALLING DEPENDENCIES ===")
!pip install -r cloud/kaggle/requirements_kaggle.txt


In [ ]:
print("\n=== RUNNING TESTS ===")
!python -m pytest -q --basetemp=/kaggle/temp tests/


In [ ]:
print("\n=== BOUNDED BENCHMARK (2,000 steps) ===")
# We benchmark M1 briefly on CPU vs GPU to determine fastest device
import time
import torch
import warnings
warnings.filterwarnings('ignore')

from sb3_contrib import MaskablePPO
from rl_v3.phase_c2_env import PhaseC2Env, PhaseC2EndpointGenerator
from tools.verification.r2_pb_wrapper import PotentialShapingWrapper
import json

with open("configs/rl_v3_phase_c2.json") as f:
    config = json.load(f)
    
gen = PhaseC2EndpointGenerator(seed=42)
gen.set_active_sizes([15])
env = PhaseC2Env(config, mode="train", generator=gen)
env = PotentialShapingWrapper(env, gamma=config["reward"]["gamma"], lambda_=config["reward"]["lambda_"])

devices = ["cpu"]
if torch.cuda.is_available():
    devices.append("cuda")

times = {}
for d in devices:
    print(f"Benchmarking M1 on {d}...")
    model = MaskablePPO("MultiInputPolicy", env, device=d, n_steps=2000, batch_size=64)
    start = time.time()
    model.learn(total_timesteps=2000)
    elapsed = time.time() - start
    times[d] = elapsed
    print(f"{d} took {elapsed:.2f}s")

best_device = min(times, key=times.get)
print(f"Optimal device: {best_device}")


In [ ]:
print(f"\n=== LAUNCHING PHASE C2: {MODEL_TO_RUN} ===")
resume_flag = "--resume" if RESUME else ""
!python cloud/kaggle/phase_c2_kaggle_runner.py --model {MODEL_TO_RUN} --interactions {MAX_INTERACTIONS} {resume_flag}

print("\n=== INVENTORY GENERATION ===")
!find /kaggle/working/uav_phase_c2 -type f -exec sha256sum {} + > /kaggle/working/inventory.txt
!cat /kaggle/working/inventory.txt
